# What we want this dataset to look like :

---

In [ ]:
import numpy as np
import xarray as xr
import zarr

# -----------------------------
# Parameters
# -----------------------------
zarr_path = "./latent_forecast.zarr"   # change to s3://... later if needed

rollout_steps = 10
n_spatial = 259_200
n_feature = 1024

init_times = np.arange(
    np.datetime64("2025-07-01T00"),
    np.datetime64("2025-07-04T00"),
    np.timedelta64(6, "h"),
)

# -----------------------------
# Coordinates
# -----------------------------
lead_time = (np.arange(1, rollout_steps + 1) * 6).astype("int64")

valid_time = (
    init_times[:, None]
    + lead_time[None, :] * np.timedelta64(1, "h")
)

# -----------------------------
# Coordinate-only Dataset
# -----------------------------
ds = xr.Dataset(
    coords={
        "init_time": ("init_time", init_times),
        "lead_time": ("lead_time", lead_time),
        "spatial_location": (
            "spatial_location",
            np.arange(n_spatial, dtype="int64"),
        ),
        "feature": (
            "feature",
            np.arange(n_feature, dtype="int64"),
        ),
        "valid_time": (
            ("init_time", "lead_time"),
            valid_time,
        ),
    },
    attrs={
        "description": "Aurora latent forecast dataset",
        "schema_version": "v1",
        "rollout_steps": rollout_steps,
        "temporal_semantics": "valid_time = init_time + lead_time",
    },
)

# -----------------------------
# Write metadata only
# -----------------------------
ds.to_zarr(
    zarr_path,
    zarr_format=3,
    mode="w",
    consolidated=False,
    write_empty_chunks=False,   # critical
)

# -----------------------------
# Create empty array (schema only)
# -----------------------------
zarr.create_array(
    zarr_path,
    name="latent_forecast",
    shape=(
        len(init_times),
        len(lead_time),
        n_spatial,
        n_feature,
    ),
    dtype="float32",
    fill_value=np.nan,
    dimension_names=(
        "init_time",
        "lead_time",
        "spatial_location",
        "feature",
    ),
)

print("Zarr schema initialized (metadata-only)")

xr.open_zarr(zarr_path, zarr_format=3, consolidated=False, chunks=None)


# Source Data
---

In [ ]:
import kafou_arraylake as arraylake
import zarr
import numpy as np
import xarray as xr

repo_name = "kafou/aurora-era5-samples"
branch = "extend-2025"

client = arraylake.Client()
repo = client.get_repo(repo_name)
ro = repo.readonly_session(branch)
ds = xr.open_zarr(
    ro.store,
    group="samples",
    zarr_format=3,
    consolidated=False,
    chunks=None,
)

print(ds)



In [ ]:
import kafou_arraylake as arraylake
import zarr
import numpy as np
import xarray as xr

repo_name = "kafou/aurora-ecmwf-samples"
branch = "main"

client = arraylake.Client()
repo = client.get_repo(repo_name)
ro = repo.readonly_session(branch)
ds = xr.open_zarr(
    ro.store,
    group="samples",
    zarr_format=3,
    consolidated=False,
    chunks=None,
)

print(ds)



# Check 

---

In [ ]:
import numpy as np
import xarray as xr
import kafou_arraylake as arraylake

repo_name = "kafou/aurora-era5-forecast-latent-vectors-november"
branch = "main"

client = arraylake.Client()
repo = client.get_repo(repo_name)
ro = repo.readonly_session(branch)

ds = xr.open_zarr(
    ro.store,
    zarr_format=3,
    consolidated=False,
    chunks=None,
)

print(ds)

init_time = np.datetime64("2024-11-01T00:00:00")
lead_time = 6  # hours

slab = ds["lv"].sel(
    init_time=init_time,
    lead_time=lead_time,
)

value = slab.isel(spatial_location=100, feature=600).values
print(value)



In [ ]:
import kafou_arraylake as arraylake 
import xarray as xr

SOURCE_REPO = "kafou/aurora-era5-forecast-lv-6z-rollout-geo-uk"
SOURCE_BRANCH = "main"

client = arraylake.Client()
repo = client.get_repo(SOURCE_REPO)
session = repo.readonly_session(SOURCE_BRANCH)

ds_lv = xr.open_zarr(session.store, zarr_format=3, consolidated=False, chunks=None)

print(ds_lv)


In [ ]:
import numpy as np
import xarray as xr
import kafou_arraylake as arraylake
import matplotlib.pyplot as plt

repo_name = "kafou/aurora-era5-forecast-lv-6z-t4-t7-geo-uk"
branch = "main"

client = arraylake.Client()
repo = client.get_repo(repo_name)
ro = repo.readonly_session(branch)

ds = xr.open_zarr(
    ro.store,
    zarr_format=3,
    consolidated=False,
    chunks=None,
)

print(ds)

lv_t = ds["lv"].sel(time="2015-01-02T12:00:00")
# dims: (spatial_location=1024, feature=1024)

arr = (
    lv_t
    .data
    .reshape(4, 16, 16, 1024)
)



fig, axs = plt.subplots(1, 4, figsize=(12, 3))

feature = 622

for i in range(4):
    im = axs[i].imshow(arr[i, :, :, feature])
    axs[i].set_title(f"Level {i}")
    axs[i].axis("off")

fig.colorbar(im, ax=axs, shrink=0.7)
plt.show()
